# A Neural Decoder for Topological Codes — independent reproduction

This notebook organizes a traceable experiment for G. Torlai and R. G. Melko, *Neural Decoder for Topological Codes*, **Physical Review Letters 119**, 030501 (2017), [doi:10.1103/PhysRevLett.119.030501](https://doi.org/10.1103/PhysRevLett.119.030501). The paper PDF is stored in `paper/docs/`.

It does not define a second implementation of the code, data generator, RBM, Gibbs decoder, MWPM, or metrics. Those components live in `ai_qec/`; this notebook selects a configuration, invokes the project runner, validates the recorded run, and displays its artifacts.

The bundled configuration is a small smoke execution. Its output verifies the path but is not a numerical reproduction of the paper's final plots or a threshold claim.

In [ ]:
# 导入实验编排、结果读取和可视化所需的依赖。
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import yaml

# 从当前目录逐级向上查找项目根目录，确保 Notebook 可从项目子目录启动。
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'run_experiment.py').is_file():
            return candidate
    raise RuntimeError('Open the notebook from this project or one of its subdirectories.')

# 解析项目、实验配置和论文原文路径，并在执行实验前确认论文文件存在。
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'experiment.torlai_melko_2017.smoke.yaml'
PAPER_PATH = PROJECT_ROOT / 'paper' / 'docs' / 'A Neural Decoder for Topological Codes.pdf'
assert PAPER_PATH.is_file(), PAPER_PATH
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(f'Project: {PROJECT_ROOT}')
print(f'Config:  {CONFIG_PATH.relative_to(PROJECT_ROOT)}')
print(f'Paper:   {PAPER_PATH.relative_to(PROJECT_ROOT)}')

## Experiment contract

The configuration is the source of truth for lattice size, noise rate, data size, RBM size, contrastive-divergence parameters, Gibbs budget, and artifact contracts. Review it before creating a run. For a paper-scale sweep, create versioned configurations in `configs/` for each `(L, p_error, seed)` point and run the same lifecycle for each configuration.

In [ ]:
# 只展示决定物理问题、数据规模和训练过程的核心配置，便于运行前核对。
print(yaml.safe_dump({key: config[key] for key in ('qec', 'noise', 'data', 'model', 'training')}, sort_keys=False))

## Execute one immutable run

`run_experiment.py` creates a unique run directory and records the resolved configuration, data manifest, child-process logs, checkpoints, predictions, metrics, benchmark report, and run manifest. The call below performs actual data generation and training; it does not create placeholder artifacts.

In [ ]:
# 调用统一实验入口；runner 按配置选择 PyTorch 环境，并将产物写入新的 run 目录。
command = [sys.executable, 'scripts/run_experiment.py', '--config', str(CONFIG_PATH), '--project-root', str(PROJECT_ROOT)]
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
# 子进程失败时输出标准错误并立即终止，避免继续读取不完整的实验产物。
if completed.returncode:
    print(completed.stderr, file=sys.stderr)
    raise RuntimeError(f'Experiment runner failed with exit code {completed.returncode}')
# 从 runner 的完成信息中提取本次运行目录，供后续单元读取 manifest 和指标。
run_line = next(line for line in completed.stdout.splitlines() if line.startswith('Run completed: '))
RUN_DIR = Path(run_line.split(': ', 1)[1].rsplit(' (', 1)[0])
RUN_DIR

## Verify provenance and generated data

The manifest must identify an actual `toric_code_capacity` run and successful generation, training, evaluation, and benchmark steps. The dataset manifest is read from the run's recorded dataset reference.

In [ ]:
# 读取 run manifest，并验证实验成功、物理模型正确且每个流水线步骤均已完成。
run_manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
assert run_manifest['status'] == 'success'
assert run_manifest['physics_fidelity'] == 'toric_code_capacity'
assert all(step['status'] == 'success' for step in run_manifest['steps'])
# 沿 manifest 记录的引用读取数据集来源，避免误用其他运行生成的数据。
dataset_manifest_path = Path(run_manifest['dataset']['manifest'])
dataset_manifest = json.loads(dataset_manifest_path.read_text(encoding='utf-8'))
# 汇总运行 ID、样本数、数据表示、码参数和噪声参数，便于检查实验可追溯性。
{
    'run_id': run_manifest['run_id'],
    'dataset_id': dataset_manifest['dataset_id'],
    'sample_counts': dataset_manifest['sample_counts'],
    'representation': dataset_manifest['representation'],
    'code': dataset_manifest['context']['code'],
    'noise': dataset_manifest['context']['noise'],
}

## Logical-failure results

The RBM metric counts a Gibbs timeout as a decoding failure, matching the decoder's explicit no-fallback policy. `mwpm_exact` is a dependency-free exact matching reference for the small lattice; its configured defect limit is recorded in the report.

In [ ]:
# 读取评估指标和基准报告；两者均由同一 run 的测试集计算得到。
metrics = json.loads((RUN_DIR / 'metrics.json').read_text(encoding='utf-8'))['metrics']
report = json.loads((RUN_DIR / 'benchmark_report.json').read_text(encoding='utf-8'))
metrics, report

In [ ]:
# 提取 RBM Gibbs 解码器与精确 MWPM 基线的逻辑错误率及 Wilson 95% 置信区间。
labels = ['RBM Gibbs', 'exact MWPM']
rates = [report['rbm']['logical_error_rate'], report['mwpm_exact']['logical_error_rate']]
intervals = [report['rbm']['wilson_95'], report['mwpm_exact']['wilson_95']]
yerr = np.array([[rate - interval[0] for rate, interval in zip(rates, intervals)],
                 [interval[1] - rate for rate, interval in zip(rates, intervals)]])
# 绘制带非对称误差条的逻辑失败率对比图。
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, rates, yerr=yerr, capsize=5, color=['#4C78A8', '#F58518'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('logical failure rate')
ax.set_title(f"L={report['lattice_size']}, p={report['p_error']}: smoke-run comparison")
ax.grid(axis='y', alpha=.25)
plt.show()

In [ ]:
# 统计物理错误链与恢复链异或后落入四个同调扇区的样本数。
sectors = ['00', '01', '10', '11']
mwpm_counts = [report['mwpm_exact']['homology_counts'][sector] for sector in sectors]
rbm_counts = [report['rbm']['homology_counts'][sector] for sector in sectors]
x = np.arange(len(sectors))
# 并排展示 RBM 与精确 MWPM 的同调扇区分布，便于观察逻辑失败类型。
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - .18, rbm_counts, .36, label='RBM accepted recoveries', color='#4C78A8')
ax.bar(x + .18, mwpm_counts, .36, label='exact MWPM', color='#F58518')
ax.set_xticks(x, sectors)
ax.set_xlabel('homology sector of physical error XOR recovery')
ax.set_ylabel('test samples')
ax.legend()
ax.set_title('Closed-cycle homology sectors')
plt.show()

## What this run establishes

This notebook establishes that the platform can generate code-capacity data, train a joint RBM by CD-k, apply syndrome-clamped Gibbs decoding, evaluate logical failure by the homology of `error XOR recovery`, and compare on the identical test split with MWPM. The smoke configuration is deliberately too small to support a scientific performance conclusion. Paper-scale conclusions require a predefined lattice/error-rate/seed grid, sufficiently large samples, and a scalable matching backend for larger defect sets.